In [2]:
import matplotlib.pyplot as plt
import anndata
import scanpy as sc
import snapatac2 as snap
import numpy as np
import pandas as pd
import os
import scanpy.external as sce
import seaborn as sns
from sklearn.metrics import silhouette_score
import numpy as np
from scipy.stats import chi2


In [3]:
import warnings
warnings.filterwarnings("ignore")


In [ ]:
df_dmr= pd.read_csv('/data1st2/hannan_25/data/Nanopore_processV1/nanopore_08_differential/summary/dmrmerged_seg_anno_2tools_nofilter_0901.csv',index_col=0)
df_dmr


In [ ]:
for mod in ['5mC', '5hmC']:
    df_dmr_5mc = df_dmr[df_dmr['mod']==mod]
    df_dmr_5mc['Region'] = df_dmr_5mc['comparision'].str[3:6]
    df_dmr_5mc = df_dmr_5mc#[df_dmr_5mc['Region']=='AMY']
    df_dmr_5mc_expanded = df_dmr_5mc.dmr.str.split('[|_:-]', expand=True)
    df_dmr_5mc_expanded.columns = ['methtype', 'motif2', 'chr','start', 'end']
    # Expand DMRs with length <100 to 100bp centered at original center
    df_dmr_5mc_expanded['length'] = df_dmr_5mc_expanded['end'].astype(int) - df_dmr_5mc_expanded['start'].astype(int)
    df_dmr_5mc_expanded['center'] = (df_dmr_5mc_expanded['end'].astype(int) + df_dmr_5mc_expanded['start'].astype(int)) //2
    df_dmr_5mc_expanded['start_expanded'] = df_dmr_5mc_expanded['center'] - 50
    df_dmr_5mc_expanded['end_expanded'] = df_dmr_5mc_expanded['center'] + 50
    # if legthn <100, expand start = start_expanded, end = end_expanded
    df_dmr_5mc_expanded.loc[df_dmr_5mc_expanded['length']<100, 'start'] = df_dmr_5mc_expanded.loc[df_dmr_5mc_expanded['length']<100, 'start_expanded']
    df_dmr_5mc_expanded.loc[df_dmr_5mc_expanded['length']<100, 'end'] = df_dmr_5mc_expanded.loc[df_dmr_5mc_expanded['length']<100, 'end_expanded']

    df_dmr_5mc =  pd.concat([df_dmr_5mc, df_dmr_5mc_expanded], axis=1)
    df_dmr_5mc.to_csv(f'/data2st1/junyi/output/atac1112/cCRE/{mod}_annotation.csv')
    df_bed= df_dmr_5mc.loc[:,['chr', 'start', 'end']].drop_duplicates()
    df_bed['chr'] = 'chr' + df_bed['chr'].astype(str)
    df_bed.to_csv(f'/data2st1/junyi/output/atac1112/cCRE/dmr_{mod}.bed', sep='\t', header=False, index=False)


In [ ]:
df_bed.drop_duplicates(['chr', 'start', 'end'])

In [ ]:
adata_concat = snap.read_dataset('/data2st1/junyi/output/atac0627/doublet_filtered.h5ads/_dataset.h5ads')

In [ ]:
%time hm5c_mat = snap.pp.make_peak_matrix(adata_concat,peak_file='/data2st1/junyi/output/atac1112/cCRE/dmr_5hmC.bed')
hm5c_mat.write(f"output/atac1112/3REGIONS_5hmc_new.h5ads")

In [ ]:
%time m5c_mat = snap.pp.make_peak_matrix(adata_concat,peak_file='/data2st1/junyi/output/atac1112/cCRE/dmr_5mC.bed')
m5c_mat.write(f"output/atac1112/3REGIONS_5mc_new.h5ads")
adata_concat.close()

In [4]:
annodict= {}
for methtype in ['5mC', '5hmC']:
    if methtype == '5mC':
        annodict[methtype] = pd.read_csv('/data2st1/junyi/output/atac1112/cCRE/5mC_annotation.csv',index_col=0)
    else:
        annodict[methtype] = pd.read_csv('/data2st1/junyi/output/atac1112/cCRE/5hmC_annotation.csv',index_col=0)
    for region in ['AMY', 'HIP', 'PFC']:
        df_region = annodict[methtype][annodict[methtype]['Region']==region]
        df_region.loc[df_region['length']<100, 'start'] = df_region.loc[df_region['length']<100, 'start_expanded']
        df_region.loc[df_region['length']<100, 'end'] = df_region.loc[df_region['length']<100, 'end_expanded']
        df_bed = df_region.loc[:,['chr', 'start', 'end']].drop_duplicates()
        df_bed['chr'] = 'chr' + df_bed['chr'].astype(str)
        df_bed.to_csv(f'/data2st1/junyi/output/atac1112/cCRE/{methtype}_{region}.bed', sep='\t', header=False, index=False)

In [ ]:
annodict['5mC'].head()

In [ ]:
for methtype in ['5mC', '5hmC']:
    methtypeL = methtype.lower()
    adata = sc.read_h5ad(f"output/atac1112/3REGIONS_{methtypeL}_new.h5ads")
    dups = adata.var_names.duplicated()
    # Drop duplicated genes
    dmr_mat = adata[:, ~dups].copy()
    #dmr_mat = dmr_mat.obs[~((dmr_mat.obs['celltype.L1_ct']=='OPC') & (dmr_mat.obs['Neurotransmitter_celltype']!='NN'))]
    dmr_mat = dmr_mat[~((dmr_mat.obs['celltype.L1_ct']=='OPC') & (dmr_mat.obs['Neurotransmitter_celltype']!='NN'))]
    adata_dict={}
    for region in ['AMY', 'HIP', 'PFC']:
        df_anno = annodict[methtype]
        df_region = df_anno[df_anno['Region']==region]
        df_region['dmr_name'] = 'chr'+df_region['chr'].astype(str)+':'+df_region['start'].astype(str)+'-'+df_region['end'].astype(str)
        adata_region = dmr_mat[dmr_mat.obs['Region']==region,]
        df_dmr_ano = df_region.drop_duplicates(subset=['dmr_name'])
        df_dmr_ano.drop('gene',axis=1,inplace=True)
        df_dmr_ano.set_index('dmr_name', inplace=True)
        adata_region = adata_region[:, df_dmr_ano.index]
        adata_region.var= df_dmr_ano.loc[adata_region.var_names, :]
        adata_region.var['chr'] = "chr"+adata_region.var['chr'].astype(str)
        #print(region, df_region.shape[0])
        adata_region.write_h5ad(f"/data2st1/junyi/output/atac1112/subset/dmr_region_nt/{region}_{methtype}.h5ad")


In [ ]:
def g_test_row(row):
    row = np.array(row, dtype=float)
    total = row.sum()
    expected = np.repeat(total/len(row), len(row))

    # Avoid log(0)
    row_safe = np.where(row > 0, row, 1e-12)

    G = 2 * np.sum(row_safe * np.log(row_safe / expected))
    pval = 1 - chi2.cdf(G, df=len(row)-1)
    return pval

for methtype in ['5mC', '5hmC']:
    for region in ['AMY', 'HIP', 'PFC']:
        adata_region = sc.read_h5ad(f"/data2st1/junyi/output/atac1112/subset/dmr_region_nt/{region}_{methtype}.h5ad")
        adata_region.layers['counts'] = adata_region.X
        adata_region.X = adata_region.layers['counts']
        sc.pp.normalize_total(adata_region, target_sum=1e6)
        #sc.pp.log1p(adata_region)
        group_key = "celltype.L1_ct"  # 替换为 obs 的列名，例如 "cell_type"
        agg_adata = sc.get.aggregate(adata_region,by=group_key,func='mean')
        agg_adata.X = agg_adata.layers['mean']
        # # softmax of over 9 classes
        # from scipy.special import softmax
        X = agg_adata.X.T
        X_norm = X / X.sum(axis=1, keepdims=True)
        X_norm = np.nan_to_num(X_norm, nan=1/9)
        df_xnorm = pd.DataFrame(X_norm, index=agg_adata.var_names, columns=agg_adata.obs_names)
        df_xnorm_safe = df_xnorm.replace(0, 1e-12)
        entropy_values = -np.sum(df_xnorm_safe * np.log(df_xnorm_safe), axis=1)
        # 放回数据框
        df_xnorm['entropy'] = entropy_values
        # pvals = np.array([g_test_row(row) for row in agg_adata.X.T])
        # df_xnorm['pval'] = pvals
        df_xnorm.to_csv(f"/data2st1/junyi/output/atac1112/subset/dmr_region_nt/{region}_{methtype}_celltype_fraction.csv")
        #
        sc.pp.log1p(adata_region)
        sc.tl.rank_genes_groups(adata_region, groupby=group_key, method="wilcoxon",pts=True)
        for ct in adata_region.obs[group_key].unique():
            df_cts = sc.get.rank_genes_groups_df(adata_region, group=ct,pval_cutoff=0.05)
            df_cts.to_csv(f'/data2st1/junyi/output/atac1112/dar/cts/dmr_wilcoxon/{region}_{methtype}_wilcox_{ct}.csv', index=False)


In [7]:
# deconvolution
df_hip_5mc = annodict['5mC'][annodict['5mC']['Region']=='HIP']

In [8]:
df_hip_5mc

,mod,motif,dmr,ifdifferent,score,num_sites,effect_size,case1_sig,case2_sig,diff.Methy,...,Region,methtype,motif2,chr,start,end,length,center,start_expanded,end_expanded
FC-HIP_vs_FW-HIP:methylDMR.157575,5mC,CG,5mC|CG_1:100032554-100033199,different,26.653646,7,0.258089,0.381571,0.123482,0.258089,...,HIP,5mC,CG,1,100032554,100033199,645,100032876,100032826,100032926
FC-HIP_vs_FW-HIP:methylDMR.157576,5mC,CG,5mC|CG_1:100213346-100213354,different,-9.883682,3,-0.144257,0.478557,0.622815,-0.144257,...,HIP,5mC,CG,1,100213300,100213400,8,100213350,100213300,100213400
FC-HIP_vs_FW-HIP:methylDMR.157577,5mC,CG,5mC|CG_1:100243374-100243516,different,-22.914126,6,-0.268296,0.336613,0.604909,-0.268296,...,HIP,5mC,CG,1,100243374,100243516,142,100243445,100243395,100243495
FC-HIP_vs_FW-HIP:methylDMR.157578,5mC,CG,5mC|CG_1:100361077-100361171,different,-9.144186,3,-0.122090,0.437606,0.559696,-0.122090,...,HIP,5mC,CG,1,100361074,100361174,94,100361124,100361074,100361174
FC-HIP_vs_FW-HIP:methylDMR.157579,5mC,CG,5mC|CG_1:100369366-100369577,different,-24.373712,8,-0.103544,0.046788,0.150332,-0.103544,...,HIP,5mC,CG,1,100369366,100369577,211,100369471,100369421,100369521
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
635385,5mC,CN,5mC|CN_Y:90803364-90803367,different,60.317453,2,-0.249814,0.001720,0.004218,-0.002498,...,HIP,5mC,CN,Y,90803315,90803415,3,90803365,90803315,90803415
635390,5mC,CN,5mC|CN_Y:90805215-90805218,different,34.261738,2,0.183754,0.008120,0.006283,0.001838,...,HIP,5mC,CN,Y,90805166,90805266,3,90805216,90805166,90805266
635391,5mC,CN,5mC|CN_Y:90805215-90805218,different,34.261738,2,0.183754,0.008120,0.006283,0.001838,...,HIP,5mC,CN,Y,90805166,90805266,3,90805216,90805166,90805266
635396,5mC,CN,5mC|CN_Y:90807478-90807486,different,50.554325,4,0.188461,0.008205,0.006321,0.001885,...,HIP,5mC,CN,Y,90807432,90807532,8,90807482,90807432,90807532


In [5]:
df_hip_atac_frac = pd.read_csv('/data2st1/junyi/output/atac1112/subset/dmr_region_nt/HIP_5mC_celltype_fraction.csv', index_col=0)

In [11]:
df_hip_5mc['extended_dmr'] = 'chr' + df_hip_5mc['chr'].astype(str) + ':' + df_hip_5mc['start_expanded'].astype(str) + '-' + df_hip_5mc['end_expanded'].astype(str)

In [12]:
df_hip_5mc['id']=df_hip_5mc.index

In [13]:
df_hip_5mc.set_index('extended_dmr', inplace=True)

In [14]:
df_merged_region = pd.merge(df_hip_5mc, df_hip_atac_frac, left_on='extended_dmr', right_index=True, how='inner')

In [15]:
X = df_merged_region.loc[:, ['Astrocyte', 'Epen', 'GABA', 'Glut', 'Microglia', 'OPC', 'Oligo', 'Vascular']].values

In [16]:
df_merged_region

,mod,motif,dmr,ifdifferent,score,num_sites,effect_size,case1_sig,case2_sig,diff.Methy,...,id,Astrocyte,Epen,GABA,Glut,Microglia,OPC,Oligo,Vascular,entropy
extended_dmr,,,,,,,,,,,,,,,,,,,,,
chr1:100213300-100213400,5mC,CG,5mC|CG_1:100213346-100213354,different,-9.883682,3,-0.144257,0.478557,0.622815,-0.144257,...,FC-HIP_vs_FW-HIP:methylDMR.157576,0.011461,0.000000,0.056092,0.032387,0.003065,0.000000,0.000000,0.896995,0.439141
chr1:100361074-100361174,5mC,CG,5mC|CG_1:100361077-100361171,different,-9.144186,3,-0.122090,0.437606,0.559696,-0.122090,...,FC-HIP_vs_FW-HIP:methylDMR.157578,0.334355,0.000000,0.132900,0.108082,0.076719,0.277279,0.070665,0.000000,1.614891
chr1:10068137-10068237,5mC,CG,5mC|CG_1:10068185-10068190,different,-8.476260,3,-0.145969,0.258978,0.404948,-0.145969,...,FC-HIP_vs_FW-HIP:methylDMR.157580,0.101346,0.000000,0.318495,0.121970,0.092019,0.285639,0.080532,0.000000,1.633347
chr1:10107851-10107951,5mC,CG,5mC|CG_1:10107899-10107904,different,-16.184733,4,-0.228307,0.190726,0.419033,-0.228307,...,FC-HIP_vs_FW-HIP:methylDMR.157581,0.087173,0.000000,0.066218,0.234425,0.015893,0.480504,0.115787,0.000000,1.400154
chr1:10112976-10113076,5mC,CG,5mC|CG_1:10113007-10113046,different,-9.050314,3,-0.090713,0.509330,0.600044,-0.090713,...,FC-HIP_vs_FW-HIP:methylDMR.157582,0.522559,0.000000,0.229556,0.136326,0.005772,0.024946,0.080842,0.000000,1.273791
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
chrX:170019104-170019204,5mC,CN,5mC|CN_X:170019147-170019162,different,-0.364068,6,0.009196,0.006320,0.006228,0.000092,...,635344,0.111111,0.111111,0.111111,0.111111,0.111111,0.111111,0.111111,0.111111,1.953089
chrY:90805166-90805266,5mC,CN,5mC|CN_Y:90805215-90805218,different,34.261738,2,0.183754,0.008120,0.006283,0.001838,...,635390,0.237636,0.000000,0.216901,0.265305,0.117891,0.075898,0.086370,0.000000,1.684279
chrY:90805166-90805266,5mC,CN,5mC|CN_Y:90805215-90805218,different,34.261738,2,0.183754,0.008120,0.006283,0.001838,...,635391,0.237636,0.000000,0.216901,0.265305,0.117891,0.075898,0.086370,0.000000,1.684279


In [17]:
Y = df_merged_region.loc[:, ['diff.Methy']].values

In [ ]:
from sklearn.linear_model import Ridge
# u

model = Ridge(alpha=1.0, positive=True)
model.fit(X, Y)

beta = model.coef_
y_pred = model.predict(X)

In [19]:
from scipy.stats import pearsonr
r, p = pearsonr(Y.flatten(), y_pred.flatten())



In [ ]:
from sklearn.linear_model import Ridge, Lasso
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score
from scipy.optimize import nnls
from sklearn.model_selection import cross_val_score

# 数据准备，X 和 Y 是已经准备好的数据
# X: 特征矩阵, Y: 目标值 (delta 5hmC/5mC 或其它)

# 模型训练
model_ridge = Ridge(alpha=1.0, positive=True)
model_ridge.fit(X, Y)

model_lasso = Lasso(alpha=0.01)
model_lasso.fit(X, Y)

model_svr = SVR(kernel='rbf', C=1.0, epsilon=0.1)
model_svr.fit(X, Y)


ValueError: Expected a one-dimensional array (vector), but the shape of b is (284077, 1)

In [27]:

#model_nnls = nnls(X, Y)

# 预测
ridge_pred = model_ridge.predict(X)
lasso_pred = model_lasso.predict(X)
svr_pred = model_svr.predict(X)
#nnls_pred = model_nnls[0] @ X.T  # NNLS 预测结果


NameError: name 'model_nnls' is not defined

In [28]:
nnls_pred = svr_pred

In [ ]:

# 计算性能指标
mse_ridge = mean_squared_error(Y, ridge_pred)
r2_ridge = r2_score(Y, ridge_pred)

mse_nnls = mean_squared_error(Y, nnls_pred)
r2_nnls = r2_score(Y, nnls_pred)

mse_svr = mean_squared_error(Y, svr_pred)
r2_svr = r2_score(Y, svr_pred)

mse_lasso = mean_squared_error(Y, lasso_pred)
r2_lasso = r2_score(Y, lasso_pred)

print(f"Ridge MSE: {mse_ridge}, R²: {r2_ridge}")
print(f"NNLS MSE: {mse_nnls}, R²: {r2_nnls}")
print(f"SVR MSE: {mse_svr}, R²: {r2_svr}")
print(f"Lasso MSE: {mse_lasso}, R²: {r2_lasso}")

# 使用交叉验证
cv_ridge = cross_val_score(Ridge(alpha=1.0), X, Y, cv=5, scoring='neg_mean_squared_error')
print(f"Ridge CV MSE: {-cv_ridge.mean()}")

cv_svr = cross_val_score(SVR(kernel='rbf', C=1.0, epsilon=0.1), X, Y, cv=5, scoring='neg_mean_squared_error')
print(f"SVR CV MSE: {-cv_svr.mean()}")

cv_lasso = cross_val_score(Lasso(alpha=0.01), X, Y, cv=5, scoring='neg_mean_squared_error')
print(f"Lasso CV MSE: {-cv_lasso.mean()}")


Ridge MSE: 0.01410353381265968, R²: 0.010040458204092939
NNLS MSE: 0.013740613753871731, R²: 0.035514653528371576
SVR MSE: 0.013740613753871731, R²: 0.035514653528371576
Lasso MSE: 0.014246575963169318, R²: 0.0
Ridge CV MSE: 0.014195966686495271
